# EDA — espaces_verts.csv
Score de Vivabilité · Inventaire des espaces verts parisiens

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import csv

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

## 1. Chargement

> ⚠️ Ce fichier contient des champs GeoJSON volumineux. On utilise `quoting=csv.QUOTE_NONE` pour contourner la limite de taille pandas.

In [ ]:
FILE = 'architecture-data/brute/score_de_vivabilité/espaces_verts.csv'

df = pd.read_csv(
    FILE,
    sep=None,
    engine='python',
    on_bad_lines='skip',
    quoting=csv.QUOTE_NONE,
    escapechar='\\'
)
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 2. Infos générales

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Valeurs manquantes — critique pour ce dataset

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, max(3, len(df.columns) * 0.45)))
colors = ['#D85A30' if v > 50 else '#BA7517' if v > 20 else '#1D9E75' for v in missing.values]
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel('% manquant')
ax.set_title('Valeurs manquantes par colonne (rouge > 50%, orange > 20%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.axvline(50, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

display(missing.rename('% manquant').to_frame())

## 4. Doublons

In [ ]:
n_dup = df.duplicated().sum()
print(f'Doublons : {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    display(df[df.duplicated(keep=False)].head(10))

## 5. Détection colonnes clés

In [ ]:
def find_col(df, *keywords):
    for kw in keywords:
        match = [c for c in df.columns if kw.lower() in c.lower()]
        if match: return match[0]
    return None

col_map = {
    'nom'           : find_col(df, 'nom', 'name', 'libelle'),
    'type'          : find_col(df, 'type', 'categorie', 'nature', 'typsite'),
    'surface'       : find_col(df, 'surface', 'superficie', 'area'),
    'arrondissement': find_col(df, 'arrond', 'arrdt', 'arr'),
    'adresse'       : find_col(df, 'adresse', 'address', 'rue'),
    'geo_point'     : find_col(df, 'geo_point', 'geopoint', 'coord'),
    'geo_shape'     : find_col(df, 'geo_shape', 'geometry', 'geom'),
}

print('Colonnes détectées :')
for k, v in col_map.items():
    status = '✅' if v else '❌'
    print(f'  {status}  {k:20s} → {v}')
print(f'\nColonnes non mappées : {[c for c in df.columns if c not in col_map.values()]}')

## 6. Répartition par type d'espace vert

In [ ]:
type_col = col_map['type']
if type_col:
    vc = df[type_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, max(4, len(vc) * 0.4)))
    ax.barh(vc.index.astype(str), vc.values, color='#1D9E75')
    ax.set_title(f"Top 20 types d'espaces verts ({type_col})")
    ax.set_xlabel("Nombre d'espaces")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('❌ Colonne type non trouvée — vérifier df.columns')

## 7. Répartition par arrondissement

In [ ]:
arr_col = col_map['arrondissement']
if arr_col:
    print(f'Complétude arrondissement : {df[arr_col].notna().sum()}/{len(df)} ({df[arr_col].notna().mean()*100:.1f}%)')
    vc_arr = df[arr_col].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(vc_arr.index.astype(str), vc_arr.values, color='#1D9E75')
    ax.set_title("Espaces verts par arrondissement")
    ax.set_xlabel('Arrondissement')
    ax.set_ylabel("Nombre d'espaces")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('❌ Colonne arrondissement non trouvée')

## 8. Distribution des surfaces

In [ ]:
surf_col = col_map['surface']
if surf_col:
    df[surf_col] = pd.to_numeric(df[surf_col], errors='coerce')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df[surf_col].dropna(), bins=40, color='#1D9E75', edgecolor='white')
    axes[0].set_title('Distribution des surfaces (m²)')
    axes[0].set_xlabel('Surface (m²)')

    axes[1].hist(np.log1p(df[surf_col].dropna()), bins=40, color='#378ADD', edgecolor='white')
    axes[1].set_title('Distribution log1p(surface)')
    axes[1].set_xlabel('log1p(m²)')

    plt.suptitle('Distribution des surfaces des espaces verts', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(df[surf_col].describe())
else:
    print('❌ Colonne surface non trouvée')

## 9. Cardinalité des colonnes (hors géo)

In [ ]:
geo_cols = [c for c in df.columns if any(k in c.lower() for k in ['geo', 'shape', 'geom'])]
non_geo  = [c for c in df.select_dtypes(include='object').columns if c not in geo_cols]

cardinality = pd.DataFrame({
    'colonne'    : non_geo,
    'unique'     : [df[c].nunique() for c in non_geo],
    'exemple'    : [str(df[c].dropna().iloc[0])[:50] if not df[c].dropna().empty else 'N/A' for c in non_geo],
    '% manquant' : [round(df[c].isnull().mean() * 100, 1) for c in non_geo],
}).sort_values('unique', ascending=False)

display(cardinality)
print(f'\nColonnes géo exclues : {geo_cols}')

## 10. Résumé + plan Silver

In [ ]:
print('=' * 55)
print('RÉSUMÉ EDA — espaces_verts.csv')
print('=' * 55)
print(f'  Lignes              : {len(df):,}')
print(f'  Colonnes            : {len(df.columns)}')
print(f'  Doublons            : {df.duplicated().sum()}')
missing_count = (df.isnull().sum() > 0).sum()
print(f'  Cols avec NaN       : {missing_count}')
print()
print('  COLONNES CLÉS DÉTECTÉES :')
for k, v in col_map.items():
    print(f'    {k:20s} → {v}')
print('=' * 55)
print()
print('ACTIONS SILVER REQUISES :')
print('  → Extraire lat/lon depuis geo_point ("lat, lon" → 2 colonnes float)')
print('  → Convertir surface en float, traiter valeurs aberrantes')
print('  → Standardiser les types (casse, accents)')
print('  → Score = surface totale espaces verts / population par arrond')
print('  → Bonus parcs > 1 ha (accessibilité piétonne réelle)')
print('  → Export Parquet pour Gold scoring vivabilité')